<a href="https://colab.research.google.com/github/awa-easymoneysniper/OSU-Mania-Auto-Beatmap-Generator/blob/main/save_mania_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install datasets

In [ ]:
from huggingface_hub import login

login()

In [ ]:
!pip -q install datasets soundfile tqdm

In [ ]:
import os
import json
import soundfile as sf

from tqdm import tqdm
from datasets import load_dataset

SAVE_ROOT = "/content/drive/MyDrive/osu_mania_dataset"

AUDIO_DIR = os.path.join(SAVE_ROOT, "audio")
META_DIR = os.path.join(SAVE_ROOT, "metadata")
INDEX_PATH = os.path.join(SAVE_ROOT, "index.json")
STATE_PATH = os.path.join(SAVE_ROOT, "state.json")  # tracks raw stream position

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

# ============================
# Resume setup
# ============================
# raw_seen = how many samples we've pulled from the stream total (includes
# non-mania samples that got skipped). This is what ds.skip() needs.
# song_id = how many mania songs we've actually saved. Independent counter.

if os.path.exists(STATE_PATH):
    with open(STATE_PATH, "r", encoding="utf-8") as f:
        state = json.load(f)
    raw_seen = state["raw_seen"]
else:
    # One-time manual override: set this to the last tqdm "it" count you saw
    # before the crash. After this run, state.json takes over automatically.
    raw_seen = 0

if os.path.exists(INDEX_PATH):
    with open(INDEX_PATH, "r", encoding="utf-8") as f:
        index = json.load(f)
else:
    index = []

song_id = len(index)

print(f"Resuming: skipping {raw_seen} raw samples, {song_id} songs already saved.")

ds = load_dataset(
    "project-riz/osu-beatmaps",
    "original",
    split="train",
    streaming=True,
)

ds = ds.cast_column("mp3", Audio(decode=False))

# Fast-forward the stream past everything we've already consumed.
if raw_seen > 0:
    print("before skip")
    ds = ds.skip(raw_seen)
    print("after skip")


def save_state():
    """Persist both the index and raw stream position, so a crash can
    resume from exactly here rather than re-scanning from the start."""
    with open(INDEX_PATH, "w", encoding="utf-8") as f:
        json.dump(index, f, indent=2)
    with open(STATE_PATH, "w", encoding="utf-8") as f:
        json.dump({"raw_seen": raw_seen}, f)


# ============================
# Process
# ============================

errors = 0

print("before loop")

for i, sample in enumerate(tqdm(ds, initial=raw_seen), start=raw_seen):

    print("got sample", i)

    raw_seen += 1

    try:
        metadata = sample["json"]

        mania_maps = []
        for bm in metadata["beatmaps"]:
            if bm["mode"] != 3:
                continue
            mania_maps.append({
                "beatmap_id": bm["beatmap_id"],
                "beatmapset_id": bm["beatmapset_id"],
                "difficulty_name": bm["version"],
                "star_rating": bm["difficultyrating"],
                "keys": bm["diff_size"],
                "bpm": bm["bpm"],
                "length": bm["hit_length"],
                "notes": bm["count_normal"],
                "long_notes": bm["count_slider"],
                "osu": bm["content"]
            })

        if len(mania_maps) == 0:
            continue

        filename = f"{song_id:06d}"

        # --------------------------
        # Save audio
        # --------------------------
        audio = sample["mp3"]

        audio_path = os.path.join(AUDIO_DIR, filename + ".mp3")

        with open(audio_path, "wb") as f:
            f.write(audio["bytes"])

        # --------------------------
        # Save metadata
        # --------------------------
        meta = {
            "audio_file": filename + ".mp3",
            "beatmaps": mania_maps
        }

        with open(os.path.join(META_DIR, filename + ".json"), "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False)

        index.append({
            "id": filename,
            "audio_file": filename + ".mp3",
            "num_charts": len(mania_maps)
        })

        song_id += 1

        if song_id % 100 == 0:
            save_state()

    except Exception as e:
        # A single corrupted/undecodable sample no longer kills the run.
        errors += 1
        print(f"Skipped a bad sample (raw #{raw_seen}): {e}")
        continue

save_state()
print(f"Finished! Saved {song_id} songs total, {errors} errors this run.")

Resuming: skipping 0 raw samples, 0 songs already saved.


Resolving data files:   0%|          | 0/205 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/205 [00:00<?, ?it/s]

before loop


0it [00:00, ?it/s]

...

15534it [1:23:25,  1.43it/s]

got sample 15533


15535it [1:23:26,  1.54it/s]

got sample 15534


15536it [1:23:26,  1.69it/s]